In [ ]:
import pandas as pd
from pymongo import MongoClient

# Connect and load 100,000 records for testing
client = MongoClient("mongodb://localhost:27017/")
db = client["nyc_taxi_analytics"]
data = list(db.december_2025.find().limit(1000000))
df = pd.DataFrame(data)

df.head() # View the first 5 rows

In [3]:
# Remove trips with 0 miles or 0 minute duration
df = df[(df['trip_miles'] > 0) & (df['trip_time'] > 0)]

In [ ]:
df.info()

In [ ]:
# Check for missing values in your DataFrame
print(df.isnull().sum())

# Drop rows where essential columns are missing
df.dropna(subset=['pickup_datetime', 'trip_miles', 'base_passenger_fare'], inplace=True)

In [8]:
import pandas as pd
import numpy as np

# Assuming df is your loaded DataFrame
# 1. Convert to datetime if not already done
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# 2. Extract base features
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek

# 3. Vectorized 'is_night' (Fastest way)
# Night defined as 8 PM (20) to 5 AM (5)
df['is_night'] = ((df['hour'] >= 20) | (df['hour'] <= 5)).astype(int)

# 4. Calculate Trip Speed (Miles per Hour)
# Avoid division by zero using np.where
df['avg_speed_mph'] = np.where(
    df['trip_time'] > 0, 
    (df['trip_miles'] / (df['trip_time'] / 3600)), 
    0
)

In [ ]:
# Display the first 5 rows focusing on the new features
cols_to_show = ['pickup_datetime', 'hour', 'day_of_week', 'is_night', 'tips']
print(df[cols_to_show].head())

In [9]:
# Encoding time as coordinates on a circle
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

In [ ]:
# Check percentage of missing data
missing_pct = df.isnull().mean() * 100
print(f"Missing Data:\n{missing_pct[missing_pct > 0]}")

# Fill missing categorical values with 'Unknown' for the model
df['originating_base_num'] = df['originating_base_num'].fillna('Unknown')

In [ ]:
# Downcast floats and ints
df['trip_miles'] = pd.to_numeric(df['trip_miles'], downcast='float')
df['PULocationID'] = pd.to_numeric(df['PULocationID'], downcast='integer')

print(f"New Memory Usage: {df.memory_usage().sum() / 1e6:.2f} MB")

In [ ]:
hourly_stats = df.groupby('hour').agg({
    'tips': 'mean',
    'trip_miles': 'mean',
    '_id': 'count'
}).rename(columns={'_id': 'trip_count'})

print(hourly_stats.sort_values(by='tips', ascending=False).head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style
sns.set_theme(style="whitegrid")

# Plot trip counts by hour
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='hour', palette='viridis')
plt.title('Distribution of Trips by Hour')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Number of Trips')
plt.show()